##Puxando as informações do RNV

**Biblioteca que permite trabalhar com Excel e importando o pandas**

In [ ]:
!pip install pandas openpyxl
import pandas as pd


**Puxando o arquivo do Excel**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

relatorio_nascidos_vivos = '/content/drive/MyDrive/PRIVATE_DATA/RNV_e_Natimorto_Total.xlsx'

df_combined = pd.read_excel(relatorio_nascidos_vivos)

In [ ]:
df_combined = df_combined.sort_values(by='NºDNV')
df_combined

**Retirando as colunas que não tem nome** (observamos que todas tem)

In [ ]:
df_combined = df_combined.dropna(subset=['NOME'])
df_combined

##Agora vamos padronizar os nomes do RNV##

**Usaremos a biblioteca unicode para transformar em minusculo, trocar ç por c, acentuação e acentos**

In [ ]:
!pip install unidecode
import unidecode

**Colocando em minusculo, tirando espaços em brancos e trocando ç por c**
**retirando g1-g9**
**retirando gemelar1-gemelar9**

In [ ]:
df_combined['NOME'] = df_combined['NOME'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)
df_combined['NOME'] = df_combined['NOME'].str.replace(r'g[0-9]+$', '', regex=True)
df_combined['NOME'] = df_combined['NOME'].str.replace(r'gemelar[0-9]+$', '', regex=True)

df_combined


**Aqui podemos colocar uma coluna Gêmeo para organizar a contagem**

In [ ]:
df_combined['GEMELAR'] = df_combined.duplicated(subset=['NOME'], keep=False).astype(int)

df_combined

**Verifiquei que tem nomes como cancelada, desconhecido, estou retirando isso**

In [ ]:
df_combined = df_combined[~df_combined['NOME'].str.contains('cancelada|cancelado|desconhecida|desconhecido', case=False)]
df_combined


**Padronizando as datas**

In [ ]:
df_combined['DATA NASC.'] = pd.to_datetime(df_combined['DATA NASC.'], format='%d/%m/%Y', errors='coerce')
df_combined = df_combined.dropna(subset=['DATA NASC.'])
df_combined['DATA NASC.'] = df_combined['DATA NASC.'].dt.strftime('%d/%m/%Y')

df_combined

## Agora vamos mexer com os dados dos adendos de altas e relatorios

In [ ]:
import pandas as pd

# Caminho da planilha Excel
caminho_planilha2 = '/content/drive/MyDrive/PRIVATE_DATA/FINAL25_09_Adendos_Relatorios_Obitos.xlsx'

# Carregar todas as abas da planilha em um dicionário de DataFrames
df_abas2 = pd.read_excel(caminho_planilha2, sheet_name=None)

# Acessar as abas pelo nome
notas_clinicas_1 = df_abas2['Matches_Adendos_Relatorios']
notas_clinicas_2 = df_abas2['Relatorios_Sem_Matches']
notas_clinicas_3 = df_abas2['Adendos_Sem_Matches']
notas_clinicas_4 = df_abas2['Merge_Adendos_Obitos']

# Exibir os DataFrames
print(notas_clinicas_1)
print(notas_clinicas_2)
print(notas_clinicas_3)
print(notas_clinicas_4)


##Pra facilitar a vida vamos colocar todos os adendos em uma única tabela, pra isso vamos organizar os dados e as colunas

In [ ]:
# Comparar as colunas que aparecem em todos os DataFrames e identificar quais estão presentes apenas em alguns

# Obter as colunas de cada DataFrame
colunas_df1 = set(notas_clinicas_1.columns)
colunas_df2 = set(notas_clinicas_2.columns)
colunas_df3 = set(notas_clinicas_3.columns)
colunas_df4 = set(notas_clinicas_4.columns)

# Colunas comuns a todos os DataFrames
colunas_comuns = colunas_df1.intersection(colunas_df2, colunas_df3, colunas_df4)

# Colunas únicas (presentes apenas em alguns DataFrames)
colunas_unicas_df1 = colunas_df1 - colunas_comuns
colunas_unicas_df2 = colunas_df2 - colunas_comuns
colunas_unicas_df3 = colunas_df3 - colunas_comuns
colunas_unicas_df4 = colunas_df4 - colunas_comuns

# Exibir os resultados
print("Colunas comuns a todos os DataFrames:")
print(colunas_comuns)
print("\nColunas únicas do DataFrame 1 (Matches_Adendos_Relatorios):")
print(colunas_unicas_df1)
print("\nColunas únicas do DataFrame 2 (Relatorios_Sem_Matches):")
print(colunas_unicas_df2)
print("\nColunas únicas do DataFrame 3 (Adendos_Sem_Matches):")
print(colunas_unicas_df3)
print("\nColunas únicas do DataFrame 4 (Merge_Adendos_Obitos):")
print(colunas_unicas_df4)


In [ ]:
notas_clinicas_1 = df_abas2['Matches_Adendos_Relatorios']
notas_clinicas_2 = df_abas2['Relatorios_Sem_Matches']
notas_clinicas_3 = df_abas2['Adendos_Sem_Matches']
notas_clinicas_4 = df_abas2['Merge_Adendos_Obitos']

In [ ]:
colunas_para_excluir = ['UNIDADE', 'ESPECIALIDADE', 'Dt. Int', 'Nome Paciente', 'CONV', 'Similaridade Levenshtein',
                        '> 24H INT', 'Idade', 'Classificação Óbito', 'Dt. Nasc', 'Nº Pront', 'OBS', 'Unnamed: 13', 'MUNICIPIO']

notas_clinicas_4 = notas_clinicas_4.drop(columns=colunas_para_excluir, errors='ignore')

In [ ]:
# Adicionar a nova coluna em cada DataFrame
notas_clinicas_1['Informação Relevante'] = "Adendos e Relatórios"
notas_clinicas_2['Informação Relevante'] = "Só Relatórios"
notas_clinicas_3['Informação Relevante'] = "Só Adendo"
notas_clinicas_4['Informação Relevante'] = "Obito Neonatal"

In [ ]:
notas_clinicas_1['Dt. Óbito'] = "Não se Aplica"
notas_clinicas_2['Dt. Óbito'] = "Não se Aplica"
notas_clinicas_3['Dt. Óbito'] = "Não se Aplica"

In [ ]:
# Obter os conjuntos de colunas de cada DataFrame
colunas_df1 = set(notas_clinicas_1.columns)
colunas_df2 = set(notas_clinicas_2.columns)
colunas_df3 = set(notas_clinicas_3.columns)
colunas_df4 = set(notas_clinicas_4.columns)

# Encontrar as colunas comuns a todos os DataFrames
colunas_comuns = colunas_df1.intersection(colunas_df2, colunas_df3, colunas_df4)

# Exibir as colunas comuns
print("Colunas comuns aos quatro DataFrames:")
print(colunas_comuns)


In [ ]:
# Encontrar as colunas que são únicas em cada DataFrame (ou seja, que não são comuns)
colunas_unicas_df1 = colunas_df1 - colunas_comuns
colunas_unicas_df2 = colunas_df2 - colunas_comuns
colunas_unicas_df3 = colunas_df3 - colunas_comuns
colunas_unicas_df4 = colunas_df4 - colunas_comuns

# Exibir as colunas únicas de cada DataFrame
print(colunas_unicas_df1)
print(colunas_unicas_df2)
print(colunas_unicas_df3)
print(colunas_unicas_df4)

In [ ]:
# Garantir que as colunas estejam na mesma ordem
colunas_ordenadas = notas_clinicas_1.columns

# Reordenar as colunas dos outros DataFrames
notas_clinicas_2 = notas_clinicas_2[colunas_ordenadas]
notas_clinicas_3 = notas_clinicas_3[colunas_ordenadas]
notas_clinicas_4_limpo = notas_clinicas_4[colunas_ordenadas]

# Concatenar os DataFrames
df_notas_clinicas = pd.concat([notas_clinicas_1, notas_clinicas_2, notas_clinicas_3, notas_clinicas_4])

# Exibir as primeiras linhas do DataFrame concatenado
print(df_notas_clinicas.head())

##Trabalhando no que agora chamamos de Notas Clinicas (Adendos e Relatórios)

In [ ]:
print(df_notas_clinicas)
df_notas_clinicas = df_notas_clinicas.dropna(subset=['Nome da Mãe'])

df_notas_clinicas

In [ ]:
df_notas_clinicas = df_notas_clinicas.drop(columns=['Unnamed: 0'], errors='ignore')


In [ ]:
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].apply(lambda x: unidecode.unidecode(x) if x is not None else None)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.replace(r'g[0-9]+$', '', regex=True)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.replace(r'gemelar[0-9]+$', '', regex=True)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.replace(r'\.|\-|[0-9]+', '', regex=True)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.replace(r'prontuario', '', regex=True)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.replace(r'gemelar[0-9]+', '', regex=True)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.replace(r'[^a-zA-Z\s]', '', regex=True)
df_notas_clinicas['Nome da Mãe'] = df_notas_clinicas['Nome da Mãe'].str.strip()
df_notas_clinicas


In [ ]:
df_notas_clinicas['Data de Nascimento'] = pd.to_datetime(df_notas_clinicas['Data de Nascimento'], format='%d/%m/%Y', errors='coerce')
df_notas_clinicas = df_notas_clinicas.dropna(subset=['Data de Nascimento'])
df_notas_clinicas['Data de Nascimento'] = df_notas_clinicas['Data de Nascimento'].dt.strftime('%d/%m/%Y')
df_notas_clinicas['Peso'] = pd.to_numeric(df_notas_clinicas['Peso'], errors='coerce')
df_notas_clinicas = df_notas_clinicas.reset_index(drop=True)


df_notas_clinicas

In [ ]:
print(df_notas_clinicas.info(), df_combined.info())

**Só para entender a contagem, quantos nomes da mãe único temos**

In [ ]:
df_notas_clinicas = df_notas_clinicas.sort_values(by='Nome da Mãe')
df_notas_clinicas['Nome da Mãe'].nunique()

##Vamos iniciar o Linkage Probabilistico

**Inicialmente vamos procurar exatamente o mesmo nome mantendo os dados dos RNV**

In [ ]:
df_merged = pd.merge(df_combined, df_notas_clinicas, left_on='NOME', right_on='Nome da Mãe', how='inner')

df_merged

**df_notas_clinicas_filtered** só tem os casos que não temos matches exatos

In [ ]:
df_combined_filtered = df_combined[~df_combined['NOME'].isin(df_merged['NOME'])]
df_notas_clinicas_filtered = df_notas_clinicas[~df_notas_clinicas['Nome da Mãe'].isin(df_merged['Nome da Mãe'])]

df_combined_filtered.shape,df_notas_clinicas_filtered.shape

In [ ]:
nomes_repetidos = df_merged[df_merged.duplicated(subset='NOME', keep=False)]

# Exibir as linhas com 'NOME' repetidos
nomes_repetidos

##Linkage Probabilistico

**Tô tentando usar o que a biblioteca fala aqui**


 https://recordlinkage.readthedocs.io/en/latest/guides/data_deduplication.html

In [ ]:
!pip install recordlinkage
import recordlinkage
from recordlinkage.index import Full


**Transformando as datas de nascimento no tipo data**

In [ ]:
df_notas_clinicas_filtered.rename(columns={'Nome da Mãe': 'NOME', 'Data de Nascimento': 'DATA NASC.', 'Peso': 'PESO'}, inplace=True)

df_combined_filtered['DATA NASC.'] = pd.to_datetime(df_combined_filtered['DATA NASC.'], errors='coerce')
df_notas_clinicas_filtered['DATA NASC.'] = pd.to_datetime(df_notas_clinicas_filtered['DATA NASC.'], errors='coerce')


**Deixando os Nomes no formato Strings**

In [ ]:
# Converter as colunas 'NOME' para string em ambos os DataFrames
df_combined_filtered['NOME'] = df_combined_filtered['NOME'].astype(str)
df_notas_clinicas_filtered['NOME'] = df_notas_clinicas_filtered['NOME'].astype(str)

# Verificar novamente os tipos de dados
print(df_combined_filtered['NOME'].dtype)
print(df_notas_clinicas_filtered['NOME'].dtype)


**Transfomrando peso no formato númerico**

In [ ]:
df_combined_filtered['PESO'] = pd.to_numeric(df_combined_filtered['PESO'], errors='coerce')

**Ler o texto sobre linkage**

Se não bloquear demora a vida inteira, primeiro vou bloquear em relação a data, e depois bloquear em relação ao peso

**Bloqueando só por peso**

In [ ]:
indexer = recordlinkage.Index()
indexer.block('DATA NASC.')

# Gerar os links candidatos
candidate_links = indexer.index(df_combined_filtered, df_notas_clinicas_filtered)

# Cria um objeto comparador
comparer = recordlinkage.Compare()

# Adiciona comparações para as colunas específicas
comparer.string('NOME', 'NOME', method='jarowinkler', threshold=0.90, label='nome')
comparer.date('DATA NASC.', 'DATA NASC.', label='data_nasc')
comparer.numeric('PESO', 'PESO', label='peso')
features = comparer.compute(candidate_links, df_combined_filtered, df_notas_clinicas_filtered)
matches = features[features.sum(axis=1) > 2]
indices_df_combined_filtered, indices_df_notas_clinicas = matches.index.get_level_values(0), matches.index.get_level_values(1)
matched_records_df_combined_filtered = df_combined.loc[indices_df_combined_filtered]
matched_records_df_notas_clinicas = df_notas_clinicas.loc[indices_df_notas_clinicas]

# Resetando o índice para facilitar a concatenação
matched_records_df_combined_filtered.reset_index(drop=True, inplace=True)
matched_records_df_notas_clinicas.reset_index(drop=True, inplace=True)

# Concatenando os DataFrames lado a lado para visualização
matched_df_1 = pd.concat([matched_records_df_combined_filtered, matched_records_df_notas_clinicas], axis=1)
matched_df_1


In [ ]:
indexer = recordlinkage.Index()
indexer.block('PESO')

# Gerar os links candidatos
candidate_links = indexer.index(df_combined_filtered, df_notas_clinicas_filtered)

# Cria um objeto comparador
comparer = recordlinkage.Compare()

# Adiciona comparações para as colunas específicas
comparer.string('NOME', 'NOME', method='jarowinkler', threshold=0.90, label='nome')
comparer.date('DATA NASC.', 'DATA NASC.', label='data_nasc')
comparer.numeric('PESO', 'PESO', label='peso')
features = comparer.compute(candidate_links, df_combined_filtered, df_notas_clinicas_filtered)
indices_df_combined_filtered, indices_df_notas_clinicas = matches.index.get_level_values(0), matches.index.get_level_values(1)
matches = features[features.sum(axis=1) > 2]
indices_df_combined_filtered, indices_df_notas_clinicas = matches.index.get_level_values(0), matches.index.get_level_values(1)
# Extraindo os registros correspondentes de ambos os DataFrames
matched_records_df_combined_filtered = df_combined_filtered.loc[indices_df_combined_filtered]
matched_records_df_notas_clinicas = df_notas_clinicas.loc[indices_df_notas_clinicas]

# Resetando o índice para facilitar a concatenação
matched_records_df_combined_filtered.reset_index(drop=True, inplace=True)
matched_records_df_notas_clinicas.reset_index(drop=True, inplace=True)

# Concatenando os DataFrames lado a lado para visualização
matched_df_2 = pd.concat([matched_records_df_combined_filtered, matched_records_df_notas_clinicas], axis=1)
matched_df_2

**Juntando os dois merge por bloqueio**

In [ ]:
# Converter a coluna 'DATA NASC.' para datetime nos dois DataFrames
matched_df_1['DATA NASC.'] = pd.to_datetime(matched_df_1['DATA NASC.'], errors='coerce')
matched_df_2['DATA NASC.'] = pd.to_datetime(matched_df_2['DATA NASC.'], errors='coerce')

# Identificar linhas exclusivas de matched_df_1 com base em 'Nº DNV'
somente_em_matched_df_1 = matched_df_1[~matched_df_1['NºDNV'].isin(matched_df_2['NºDNV'])]

# Identificar linhas exclusivas de matched_df_2 com base em 'Nº DNV'
somente_em_matched_df_2 = matched_df_2[~matched_df_2['NºDNV'].isin(matched_df_1['NºDNV'])]

# Combinar ambos os DataFrames exclusivos
exclusive_combined_df = pd.concat([somente_em_matched_df_1, somente_em_matched_df_2], ignore_index=True)

# Exibir o DataFrame combinado com linhas exclusivas
print(exclusive_combined_df.shape)

print(somente_em_matched_df_1.shape, somente_em_matched_df_2.shape)

In [ ]:
# Combinar matched_df_1 com as linhas exclusivas de matched_df_2
matched_df = pd.concat([matched_df_1, somente_em_matched_df_2], ignore_index=True)

# Exibir o DataFrame combinado
print(matched_df)

In [ ]:
!pip install fuzzywuzzy
from fuzzywuzzy import fuzz

# Função para calcular a similaridade de Levenshtein entre duas strings
def levenshtein_similarity(row):
    # Calcula a similaridade entre 'NOME' e 'Nome da Mãe'
    return fuzz.ratio(row['NOME'], row['Nome da Mãe']) / 100

matched_df['Levenshtein Similarity'] = matched_df.apply(lambda row: levenshtein_similarity(row), axis=1)

# Exibir o DataFrame com a nova coluna
matched_df[['NOME', 'DATA NASC.', 'PESO', 'Peso', 'Data de Nascimento', 'Nome da Mãe', 'Levenshtein Similarity']]

In [ ]:
print(matched_df.columns)
print(df_merged.columns)

In [ ]:
# Renomear a coluna no primeiro DataFrame
matched_df.rename(columns={'Levenshtein Similarity': 'Distancia de Levenshtein'}, inplace=True)

# Criar a coluna correspondente no segundo DataFrame
df_merged['Distancia de Levenshtein'] = 1


In [ ]:
df_merged = df_merged[df_merged['DATA NASC.'] == df_merged['Data de Nascimento']]
df_merged

In [ ]:
# Adicionar a coluna 'Nome Exato' com valor 'Sim' em df_merged
df_merged['Nome Exato'] = 'Sim'

# Adicionar a coluna 'Nome Exato' com valor 'Não' em matched_df
matched_df['Nome Exato'] = 'Não'

# Exibir os dataframes atualizados para verificação
print(df_merged.head())
print(matched_df.head())


In [ ]:
# Remove colunas duplicadas em matched_df
matched_df = matched_df.loc[:, ~matched_df.columns.duplicated()]

# Remove colunas duplicadas em df_merged (se houver)
df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]

In [ ]:
# Reseta o índice dos DataFrames para garantir que os índices sejam únicos
df_merged_reset = df_merged.reset_index(drop=True)
matched_df_reset = matched_df.reset_index(drop=True)

# Concatena os DataFrames
merged_final = pd.concat([df_merged_reset, matched_df_reset], ignore_index=True)

# Ordena pelo índice 'NºDNV' e reseta o índice para o DataFrame resultante
merged_final = merged_final.sort_values(by='NºDNV', ascending=True).reset_index(drop=True)

# Exibe o DataFrame resultante
print(merged_final)


In [ ]:
merged_final.columns


In [ ]:
# Identificar valores que não podem ser convertidos
invalid_data_nasc = merged_final['DATA NASC.'][pd.to_datetime(merged_final['DATA NASC.'], errors='coerce').isna()]
invalid_data_nascimento = merged_final['Data de Nascimento'][pd.to_datetime(merged_final['Data de Nascimento'], errors='coerce').isna()]

# Exibir valores inválidos
print("Valores inválidos em 'DATA NASC.':")
print(invalid_data_nasc)

print("\nValores inválidos em 'Data de Nascimento':")
print(invalid_data_nascimento)

In [ ]:
# Converter as colunas para datetime, considerando que o dia vem primeiro
merged_final['DATA NASC.'] = pd.to_datetime(merged_final['DATA NASC.'], dayfirst=True, errors='coerce')
merged_final['Data de Nascimento'] = pd.to_datetime(merged_final['Data de Nascimento'], dayfirst=True, errors='coerce')

# Calcular a diferença em dias
merged_final['Diferença em dias'] = (merged_final['DATA NASC.'] - merged_final['Data de Nascimento']).dt.days

# Exibir as primeiras linhas
print(merged_final[['DATA NASC.', 'Data de Nascimento', 'Diferença em dias']].head())


In [ ]:
diferentes_de_zero = merged_final[merged_final['Diferença em dias'] != 0]

# Exibir o DataFrame filtrado
print(diferentes_de_zero[['NOME', 'Nome da Mãe', 'DATA NASC.', 'Data de Nascimento']])
print(merged_final[['NOME', 'Nome da Mãe', 'DATA NASC.', 'Data de Nascimento']])

In [ ]:
merged_final.loc[merged_final['Diferença em dias'] != 0, 'DATA NASC.'] = merged_final['Data de Nascimento']


**Achando os relatórios que não deram matches**


In [ ]:
sem_matches = df_notas_clinicas[~df_notas_clinicas['Nome do Arquivo'].isin(merged_final['Nome do Arquivo'])]
sem_matches

In [ ]:
dnv_sem_adendo = df_combined[~df_combined['NºDNV'].isin(merged_final['NºDNV'])]

# Exibir os primeiros resultados
dnv_sem_adendo

In [ ]:
with pd.ExcelWriter('dados_resultado2.xlsx') as writer:
    # Salvando merged_final na primeira aba
    merged_final.to_excel(writer, sheet_name='Merged Final', index=False)

    # Salvando sem_matches na segunda aba
    sem_matches.to_excel(writer, sheet_name='Sem Matches', index=False)

    dnv_sem_adendo.to_excel(writer, sheet_name='DNV Sem Adendo', index=False)

# Caminho do arquivo onde os dados foram salvos
file_path = 'dados_resultado2.xlsx'
print(f"DataFrame salvo em: {file_path}")
